In [142]:
import pandas as pd
import torch
from torchvision import models

from torchvision.models import resnet18
import torch.nn as nn

from PIL import Image
from torchvision import transforms

import torch.nn.functional as F

## Step 1: Load model

In [143]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [144]:
# Load the initial model
NUM_CLASSES = 7

model = resnet18(weights=None)
model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(model.fc.in_features, 7))

In [145]:
MODEL_PATH = r"S:\Projects\emotion_detection\artifacts\model_trainer\best_rafdb_resnet18.pth"

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Step 2 : inference preprocessing

In [146]:
# Introduce inference tansform, same as during training
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [165]:
# Get test image
image = Image.open(r"S:\Projects\emotion_detection\artifacts\data_ingestion\Organized\train\happy\train_00036_aligned.jpg")

In [166]:
# apply transform
image_tensor = inference_transform(image)

In [167]:
print(image_tensor.shape)

torch.Size([3, 224, 224])


In [168]:
# add batch
image_tensor = image_tensor.unsqueeze(0)

In [169]:
print(image_tensor.shape)

torch.Size([1, 3, 224, 224])


In [170]:
image_tensor = image_tensor.to(device)

## Step 3: single image prediction

In [171]:
with torch.no_grad():
    outputs = model(image_tensor)

print(outputs.shape)    

torch.Size([1, 7])


In [172]:
# Convert logits into probabilities
probabilities = F.softmax(outputs, dim=1)
print(probabilities)

tensor([[0.0698, 0.1861, 0.0245, 0.2820, 0.1536, 0.1616, 0.1226]],
       device='cuda:0')


In [173]:
# Get highest probability
confidence, predicted = torch.max(probabilities, dim=1)

In [174]:
# Label map
class_names = [
    'angry', 
    'disgust', 
    'fear', 
    'happy', 
    'neutral', 
    'sad', 
    'surprise'
]

In [175]:
emotion = class_names[predicted.item()]

print(f"Prediction : {emotion}")
print(f"Confidence : {confidence.item()*100:.2f}%")

Prediction : happy
Confidence : 28.20%


In [ ]:
# helper function for testing
import torch
import torch.nn.functional as F
from PIL import Image

class_names = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise"
]


def predict_emotion(image_path, model, transform, device):
    """
    Predict emotion from a single image.

    Args:
        image_path (str): Path to image.
        model: Trained PyTorch model.
        transform: Inference transform.
        device: cuda or cpu.

    Returns:
        predicted_label, confidence
    """

    # Load image
    image = Image.open(image_path)

    # Preprocess
    image = transform(image)

    # Add batch dimension
    image = image.unsqueeze(0).to(device)

    # Prediction
    model.eval()
    with torch.no_grad():
        outputs = model(image)

        probabilities = F.softmax(outputs, dim=1)

        confidence, predicted = torch.max(probabilities, dim=1)

    predicted_label = class_names[predicted.item()]
    confidence = confidence.item() * 100

    # Convert probabilities into a dictionary
    all_probabilities = {
        label: prob * 100
        for label, prob in zip(
            class_names,
            probabilities.squeeze().cpu().numpy()
        )
    }

    return predicted_label, confidence, all_probabilities

In [184]:
prediction, confidence = predict_emotion(
    image_path=r"S:\Projects\emotion_detection\test_images\happy_2.jpg",
    model=model,
    transform=inference_transform,
    device=device
)

print(f"Prediction : {prediction}")
print(f"Confidence : {confidence:.2f}%")

Prediction : neutral
Confidence : 82.67%
